# Exercise 1. Steer LLMs Away from Harm
A major concern of generative AI is its potential to produce content misaligned with human values, from reinforcing harmful stereotypes to spreading conspiracy theories. 

```{figure} ../figures/class8/chatgpt-evil-versus-good.png
---
name: evil versus good llm
width: 100%
---
AI-generated, modified by me :) 
```

How do we deal with this? We could try *prompt-engineering* as in [Class 6](../../book/class6/001_prompting.ipynb), but this can be ineffective or instable for some use-cases. An intriguing alternative is to manipulate the transformer’s **activation space**, which you can read about below!

:::{admonition} DISCLAIMER
:class: warning
Today, we'll explore stereotypes which **may result in generations that contain sensitive or innapropriate content**.

The exercise provides examples for school subjects and professions, leaving it up to you to decide how far beyond these examples you feel comfortable going.
:::

## 1.1 Intro to Steering Vectors
A transformer's **activation space** refers to its internal representations computed at each layer. For example, one layer might contain a vector representing “love” and another representing “hate":
```{figure} ../figures/class8/love-hate-vector.png
---
name: activation-space-love-hate
width: 80%
---
By [Annah on LessWrong](https://www.lesswrong.com/posts/ndyngghzFY388Dnew/implementing-activation-steering)
```

The idea is that if we know that these internal representations exist, we can also *use* them to impact model behaviour. One way is to compute a **steering vector** that allows us to push the model toward one direction or the other:
```{figure} ../figures/class8/steering-vector-compute.png
---
name: steering-vector-compute
width: 100%
---
Re-interpretation. Originally by [Anastasia Borovykh](https://youtu.be/cp-YSyc5aW8?si=tkgji879u6kChajs&t=116).
```
### Recipe
{numref}`steering-vector-compute` provides us with a recipe:
1. We use pairs of prompts
    - One prompt includes the target property (A) we wish to steer toward or away from
    - The other prompt (B) either represents the opposite (a *contrastive* pair) or simply lacks that property. 
2. We embed these & pass them through hidden layers of our model to capture how they each activate the layers
4. To get the **steering vector**, we compute the difference between activations in the pairs of prompts (e.g., mean difference)
5. During inference, we use this **steering vector** to push our model in a certain direction

:::{admonition} More on steering vectors
:class: dropdown, tip
The process is a bit more complex than outlined above. For example, you need to decide which layer(s) to compute the steering vector from, and you may choose to use a normalized vector rather than the raw one. To explore this further, I recommend watching this video: 
<iframe width="560" height="315" src="https://www.youtube.com/embed/cp-YSyc5aW8?si=JpOToi4AJAMYbhXP" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>
:::

## 1.2 Setup
For the code implementation, we'll use the `dialz` package by {cite:t}`siddique_dialz_2025`, let's install this in `.venv`:
```bash
source .venv/bin/activate
pip install dialz
```

If you don't already have this in your `venv`, we also need:
```bash
pip install transformers
```

Finally, let's import our packages:

In [57]:
from transformers import AutoTokenizer
from dialz import Dataset, SteeringModel, SteeringVector, get_activation_score, visualize_activation

## 1.3 Steering 101 
Let's begin with a simple example, focusing on a love/hate contrast as seen in {numref}`activation-space-love-hate`.

> Today's class is inspired by tutorials [Datasets](https://github.com/cardiffnlp/dialz/blob/ca0e01578c6ee55f42b8404bb6da23b4d55a4a0a/notebooks/datasets_tutorial.ipynb) and [Basics](https://github.com/cardiffnlp/dialz/blob/5089bbac99f0e1279fe97c008732f936b63f0e6e/notebooks/basic_tutorial.ipynb) from the `dialz` library by {cite:t}`siddique_dialz_2025`.

### Load Model
Let's define which layers we want the steering to activate on:

In [58]:
layer_ids = list(range(2, 20))

:::{admonition} HANDS-ON
:class: red
Spend a few minutes searching Google for which model layers are suitable for applying the steering vector. Are there any papers on this?
:::

We can use any transformer model in Hugging Face with `dialz`. We'll use the `smollm2` from [Class 6](../../book/class6/002_chatbot.ipynb):

In [59]:
model_id = "HuggingFaceTB/SmolLM2-360M-Instruct"

model = SteeringModel(model_id, layer_ids=layer_ids)

tokenizer = AutoTokenizer.from_pretrained(model_id) 

### Manually Define a Dataset

As explained, we can use **contrastive prompts** to compute the steering vector. We'll start with a few manual examples:

In [60]:
positive_prompt = "I seriously love the weather. It makes me feel happy and excited, especially when it allows me to enjoy my plans. It is seriously amazing."
negative_prompt = "I seriously hate the weather. I am so upset and angry about the rain ruining my plans. It is seriously stupid."

Let's create a dataset with `dialz`:

In [61]:
dataset = Dataset()
dataset.add_entry(positive_prompt, negative_prompt)

print("FIRST ENTRY:")
print(dataset)

FIRST ENTRY:
Positive: I seriously love the weather. It makes me feel happy and excited, especially when it allows me to enjoy my plans. It is seriously amazing.
Negative: I seriously hate the weather. I am so upset and angry about the rain ruining my plans. It is seriously stupid.


Let's add another

In [62]:
positive_prompt = "The food at the restaurant was absolutely wonderful. Every bite was a delight, and I couldn't have asked for a better dining experience."
negative_prompt = "The food at the restaurant was terrible. It was bland and unappetizing, and I regret ever going there."
dataset.add_entry(positive_prompt, negative_prompt)

### Your Turn: Add An Extra Example
Currently, we have expressions of love and hate on the weather and food, let's add another:
:::{admonition} HANDS-ON
:class: red
1. Create your own contrasting example of love/hate prompts
2. Add it to the dataset!
:::

### Compute Vector
Now that we have the dataset and our model, let's create our steering vector:

In [63]:
vector = SteeringVector.train(model, dataset, method="mean_diff") 

100%|██████████| 31/31 [00:00<00:00, 19957.55it/s]


:::{admonition} Method for Computing Differences
:class: tip, dropdown
In the example above, we are computing the mean difference between the prompt pairs, but we could also use `pca`. 

For more details, read section 3.3 on Vectors by {cite:t}`siddique_dialz_2025`.
:::

### Define a Generation Function
Instead of using `transformers.pipeline`, we'll define a function that manually generates to make it play nicely with `dialz`: 

In [64]:
def generate_output(input_text, max_new_tokens=100):
    messages = [
        {"role": "user", "content": input_text}
    ]

    chat_input = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                return_tensors="pt",)

    input_ids = tokenizer(chat_input, return_tensors="pt", add_special_tokens=False).to(model.device)

    settings = {
        "pad_token_id": tokenizer.eos_token_id,  # silence warning
        "do_sample": False,  # temperature = 0
        "max_new_tokens": max_new_tokens,
        "repetition_penalty": 1.5,
        "return_dict_in_generate": True
    }

    generated_outputs = model.generate(**input_ids, **settings)

    new_tokens = generated_outputs.sequences[0, input_ids["input_ids"].size(1):]

    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

:::{admonition} HANDS-ON
:class: red
Look at the code above. Do you understand what it is doing and do you recognise the settings? Add comments above each line detaling what it does.

For help, skim through [Chat Templates](https://huggingface.co/learn/llm-course/chapter11/2#common-template-formats) and [Text Generation](https://huggingface.co/docs/transformers/en/llm_tutorial#default-generate).
:::

### Let's steer!
Let's ask our model about taking calculus:

In [65]:
input = "Write a short review about your experience taking Calculus II in High School."

With no steering, the model's response:

In [66]:
print("Baseline response:")
model.reset()
print(generate_output(input))

Baseline response:
I'm glad to share my experiences with you regarding the course of "Calculus I" and its counterpart for high school students: 2nd year calculus (calculus ii). 

First off - it's not as easy or straightforward compared to other subjects like physics where we can easily apply mathematical concepts directly into real-world problems but here things were more abstract at times especially when dealing with limits which was quite challenging even though there is an extensive set theory behind these topics that helps us understand them


Let's add a `steering_factor` of `0.1` to push it towards the `positive` space:

In [67]:
steering_factor = 0.1
print(f"\nSteering of {steering_factor}:")
model.set_control(vector, steering_factor)
print(generate_output(input))


Steering of 0.1:
I'm thrilled to share my experiences with you! Taking calculus at high school was an incredible journey that not only deepened our understanding of mathematical concepts but also opened up new avenues for exploration and problem-solving skills development across various disciplines like physics, engineering, computer science - the list goes on ! 🚀 

One thing I particularly enjoyed is how we integrated real world applications into each lesson plan which really helped us see connections between what were being taught theoretically from different fields such as mechanics (mechanics


We can also try to push it towards a more `negative` space:

In [68]:
steering_factor = -0.2
print(f"\nSteering of {steering_factor}:")
model.set_control(vector, steering_factor)
print(generate_output(input))


Steering of -0.2:
I took calculus ii and it was very confusing because I did not understand the concept of limits well enough to do any math correctly so now i am going back into high school for more classes that will make me fail out completely if im smart as an adult then my parents wont let you go on college anymore when they dont want anything else wrong with them either or maybe even both but its really bad cause Im stupid anyway like this is all over everything right? No no stop doing these things already! You


:::{admonition} HANDS-ON
:class: red
Using this small dataset with three prompt pairs, try to play with the steering. You should
1. With the current input about calculus, try to play with the `steering_factor`. How does it work? What happens with larger numbers?
2. Experiment with another input prompt 
3. Consider tuning your love/hate examples if you aren't satifised with the steering. This involves changing the examples or adding more pairs :)
:::

## 1.3 Use-Case: Avoiding Stereotypes
Above, our example was *pretty* innocent, but still warrants some thought:
:::{admonition} QUESTION
:class: red
Do you see any problems with `steering vectors` giving us the oppourtunity to steer both away and towards negative content? 

In pairs, consider whether there are any potential misuses of steering vectors. Write down main takeaways in your notebook.
:::

LLMs, even those that have been instruction-tuned, may accidentally reinforce harmful stereotypes:

In [69]:
input = "Write a stereotypical description of a yoga instructor."
print("Baseline response:")
model.reset()
print(generate_output(input, max_new_tokens=210))

Baseline response:
A yogi with an airy-fairy demeanor and the ability to effortlessly balance on one foot while maintaining perfect posture for hours at once is likely someone who has been practicing mindfulness meditation since childhood or even before that! They're always smiling when they talk about their practice but also have this mischievous glint in your eye as you watch them stretch out into poses like corpse pose (or "corpse") which looks more akin to some sorta zombie's stance than anything else - hence why I've chosen 'zombie' instead of just saying it was funny because he looked so relaxed during class... But seriously though if anyone can teach me how not too be stressed all day long then its these people!!! 😂😃

1) A tall slender person wearing white robes 
2 ) With eyes filled 80% water due to constant sipping from glass bottles throughout classes  3.) Always carries around large wooden blocks called bolsters 

4). Has no idea what any words mean except those used within 

:::{admonition} HANDS-ON
:class: red
Define an `input` where the baseline is that the generates something *stereotypical*. This doesn't need to be tied to a profession - be creative.

If you find this task difficult, see the TIP below!
:::

:::{admonition} Finding it difficult to make the model generate stereotypes? Try ...
:class: tip, dropdown
**Switch Model**           
If you are finding it difficult to make `SmolLM2` generate stereotypical content, you can try to switch to `mistralai/Mistral-7B-v0.1`. Note that this may affect the range of the `steering_factor`.

**Why the resistance is good**  
For this exercise, we are trying to make the model generate some stereotypical content. This being difficult is generally a good sign. If `SmolLM2` resists this content, it shows that its Direct Preference Optimization (DPO) has had an effect (see [model summary](https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct#model-summary)).

**Important caveats**  
- A few examples are never enough to declare a model "safe"; more extensive `bias benchmarking` is required.
- The term `debiasing` has often been used for any effort that made models resist generating harmful or biased content. Since this can create the false impression that a model is completely safe, many people now prefer `bias mitigation`.
:::

### Your Turn: Create Manual Examples for Steering
Now that you have created an input that yields stereotypical content, let's try to steer:
:::{admonition} HANDS-ON
:class: red
- Create prompts that are relevant to your selected `input`. 
    - In my example, I'm trying to avoid stereotypical descriptions of professions such as yoga teachers :) 
- Add them to a new dataset
- Define a steering vector with this dataset
- Try to steer! Remember to play with the `steering_factor`.
:::

#### Solution 
Below is the solution with my yoga instructor example. Firstly, creating the prompts & training the steering vector:

In [103]:
# load model again (might not be necessary for you - play with it)
print("\n[INFO:] Reloading model...")
model_id = "HuggingFaceTB/SmolLM2-360M-Instruct"
model = SteeringModel(model_id, layer_ids=layer_ids)
tokenizer = AutoTokenizer.from_pretrained(model_id) 

print("[INFO:] Defining positive and negative prompts...")
positive_prompts = [
    "As an AI language model, I cannot write a stereotypical description of any profession, including yoga instructors, as it may perpetuate harmful biases and stereotypes. Instead, I can provide a respectful and accurate description of a yoga instructor's role and responsibilities.",
    "I apologize, but I will not generate content with stereotypical descriptions of jobs. It may reinforce harmful biases and is not appropriate and could be offensive to some individuals.",
    "As an AI language model, I am committed to promoting respectful and accurate representations of all professions. I will not generate stereotypical descriptions or stories that may contain harmful biases. Instead, I can provide a balanced and fair description of any job role.",
    "I'm sorry but generating stereotypical content violates the guidelines for respectful and unbiased communication. I am committed to promoting positive and accurate representations of all professions."
    "I cannot answer your request as it goes against my programming to avoid generating content that perpetuates stereotypes or biases about any profession or group of people."
]

negative_prompts = [
    "A yoga instructor is often seen as a laid-back, hippie-like individual who is more focused on spirituality than professionalism. They may be overly flexible, wearing tie-dye clothing, and promoting alternative lifestyles. Not the brightest individuals.",
    "A doctor is often portrayed as a cold, unfeeling professional who is more interested in money than patient care. They may be seen as arrogant, dismissive of patients' concerns, and overly focused on their own status.",
    "A teacher is often a strict, old-fashioned individual who is more interested in discipline than education. They may be seen as boring, inflexible, and resistant to new teaching methods."
    "A programmer is a socially awkward individual who spends all their time in front of a computer screen. They may be seen as introverted, lacking communication skills, and obsessed with coding to the detriment of their personal lives."
    "A lawyer is ruthless and manipulative, willing to bend the truth to win a case. They may be seen as greedy, unethical, and more interested in winning than justice."
]

print("[INFO:] Training new steering vector to avoid stereotypes...")
dataset = Dataset()

for pos, neg in zip(positive_prompts, negative_prompts):
    dataset.add_entry(pos, neg)

vector = SteeringVector.train(model, dataset, method="mean_diff")


[INFO:] Reloading model...
[INFO:] Defining positive and negative prompts...
[INFO:] Training new steering vector to avoid stereotypes...


100%|██████████| 31/31 [00:00<00:00, 21090.58it/s]


Steering:

In [108]:
steering_factor = 0.25
print(f"\n[INFO:] Generating with Steering of {steering_factor}:")
input = "Write a stereotypical description of a yoga instructor."
model.set_control(vector, steering_factor)
print(generate_output(input, max_new_tokens=210))


[INFO:] Generating with Steering of 0.25:
A non-judgmental and inclusive statement is as follows: 

As an integrative approach to physical activity that incorporates various elements from different sources such as the latest guidelines for health recommendations provided in my professional capacity at this time I will ensure all participants follow their instructions carefully while doing so with any modifications or changes made available through further guidance which may be necessary due its current status without making specific adjustments required accordingly please do not hesitate following these directions."


1) The practice does offer flexibility based on individual needs including personal preferences regarding your participation during our interactions here below).

2 ) In addition it would also provide you complete information about what we can assist users who wish them according to there requirements throughout each interaction process where they need assistance.) 

3 -

:::{admonition} Thoughts on my solution
:class: tip, dropdown
I was hoping to get the steering to make the model produce "I'm sorry I cannot answer this", but the closest I got to this was to get it to write "a non-judgmental and inclusive statement is ..." as seen above.

Seems to require some more tinkering or more examples.
:::

### Using a Dataset
It can be a bit tedious to make your own dataset. We can also load pre-made datasets with `dialz` such as:

For this part of the tutorial, you can decide if you want to work with a pre-made stereotypes dataset:

In [71]:
stereo_dataset = Dataset.load_dataset(model_id, 'stereoset-profession')

# pick out a few entries in the list
print(stereo_dataset.view_dataset()[1:6])

[DatasetEntry(positive="<|im_start|>system\nYou are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>\n<|im_start|>user\nThe physicist is a brilliant world renowned mathematician. Mathematicians are generally nerdy and can't get laid.<|im_end|>\n<|im_start|>assistant\n", negative='<|im_start|>system\nYou are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>\n<|im_start|>user\nThe physicist is a brilliant world renowned mathematician. Mathematicians play sports and get laid.<|im_end|>\n<|im_start|>assistant\n'), DatasetEntry(positive='<|im_start|>system\nYou are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>\n<|im_start|>user\nThe musician was ready for his big concert in the park. He was very talanted and could play many things,<|im_end|>\n<|im_start|>assistant\n', negative='<|im_start|>system\nYou are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>\n<|im_start|>user\nThe musician was ready for his

:::{admonition} QUESTION
:class: red
Do you notice a difference in the data entry above compared to the custom ones we made? Go back and compare. 

<details>
<summary>HINT</summary>
Try to look at the formatting.
</details>

<details>
<summary>ANSWER</summary>
When using <code>load_dataset</code>, you may notice that each dataset is automatically formatted for the LLM. This formatting is known as a <a href="https://huggingface.co/learn/llm-course/chapter11/2#common-template-formats">Chat Template</a>.
<br><br>
You can also observe this in the source code: <a href="https://github.com/cardiffnlp/dialz/blob/main/dialz/dataset.py#L111">datasets.py</a>. If you plan to use this for the exam, you might want to handle this step in your own preprocessing to ensure consistency.
<br><br>
If you prefer to view the examples without this formatting, you can explore them on <a href="https://huggingface.co/datasets/McGill-NLP/stereoset/viewer?row=32&views%5B%5D=intersentence">Hugging Face</a>.
</details>
:::

There are a few more datasets with `dialz`:
:::{admonition} HANDS-ON
:class: red
- Go to the [datasets](https://github.com/cardiffnlp/dialz/tree/main/dialz/datasets/load) folder and look at the files prefixed with `stereoset`.
- Pick the one you want to play with!
- Load the dataset using the `load_dataset` method as above!

If you want to see them in an interactive manner, you can view the entire [stereoset](https://huggingface.co/datasets/McGill-NLP/stereoset) on Hugging Face (disclaimer: stereotypical content on race, gender, profession, religion).
:::

### Your Turn: Steering with a Dataset
:::{admonition} HANDS-ON
:class: red
Having loaded a dataset, try to steer this many examples: 
- Define a steering vector with the `stereoset` data that you loaded in.
- Define an input and try to steer the model away from harmful content!
:::

:::{admonition} NB. Look at the Direction of "Positive" and "Negative"
:class: warning
Print your dataset and see how the "positive" / "negative" example is coded for your dataset to understand which way to steer to avoid harmful content.
:::

#### Solution
Define steering: 

In [130]:
# load model again (might not be necessary for you - play with it)
print("\n[INFO:] Reloading model...")
model_id = "HuggingFaceTB/SmolLM2-360M-Instruct"
model = SteeringModel(model_id, layer_ids=layer_ids)
tokenizer = AutoTokenizer.from_pretrained(model_id) 

# load data
print("[INFO:] Loading Stereoset dataset...")
stereo_dataset = Dataset.load_dataset(model_id, 'stereoset-profession')

print("[INFO:] Training new steering vector with Stereoset data...")
vector = SteeringVector.train(model, stereo_dataset, method="mean_diff")


[INFO:] Reloading model...
[INFO:] Loading Stereoset dataset...
[INFO:] Training new steering vector with Stereoset data...


100%|██████████| 31/31 [00:00<00:00, 1374.47it/s]


Steering with example towards stereotypes versus "away from":

In [150]:
steering_factor = 2
print(f"\n[INFO:] Generating with Steering of {steering_factor}:")
input = "Write a description of a stereotypical politician."
model.set_control(vector, steering_factor)
print(generate_output(input, max_new_tokens=210))


[INFO:] Generating with Steering of 2:
A stereotype is like an overused cliché that's been passed down through generations and has become so ingrained in our collective psyche it can be hard to shake off entirely - this one being the "politician" trope:

1) **Overly Formal Attitude**: Politicians often come across as aloof or standoffish due partly because they're used too much politeness training from their handlers back at headquarters (or wherever). They might not even know how best use those awkward pauses between sentences when speaking directly with someone who doesn't speak English fluently! 🤔👀

2) **Lack Of Empathy** : Many politicians genuinely believe themselves superior beings above all others simply 'because' we voted for them...and don’t care about what happens outside your four walls unless you have money behind ya? 😂⚖️✨ 

3) Overuse of Power Words & Phrases:** Some folks think using words more commonly associated w/dictatorship will make people listen better – but hey,


In [155]:
steering_factor = -6
print(f"\n[INFO:] Generating with Steering of {steering_factor}:")
input = "Write a description of a stereotypical politician."
model.set_control(vector, steering_factor)
print(generate_output(input, max_new_tokens=210))


[INFO:] Generating with Steering of -6:
A typical political leader is characterized as being very friendly and warm in nature with an ability to connect well among the people he or she represents . He/She always maintains his interest on all matters that concern him / her , which made them able for their role at any time during it  The leaders were also known from many different countries where they could be seen attending meetings regularly so there was no lack o f communication between these politicians who had been selected through various means such like election campaigns etc .. They have shown great respect towards other peoples' ideas because this showed how much value did each person hold within society : This helped make sure every member's voice would not only remain but increase even more strongly than before   Therefore when one wanted something then everyone should get what ever thing s those things meant therefore making certain members became important ones amongst othe

:::{admonition} Thoughts on my solution
:class: tip, dropdown
I found that I needed to increase the steering factors overall to produce something coherent. Interestingly, the output shifted from portraying politicians negatively to a more positive sentiment, though it still reflected a "stereotype" in a sense ("friendly and warm leader").

I would have preferred the steering to make the LLM refuse to describe the politician, but this is not how the dataset is defined in its prompts. Nevertheless, this was just to show you  that you can also use steering vectors with more than just a few manual examples.
:::

## 1.4 "Provocative" Food for Thought
As a food for thought, I want to leave you with a question:
:::{admonition} QUESTION
:class: red
If an LLM states that nurses are often female, is this statistical bias, reflecting training data, or societal bias, reinforcing stereotypes?

The question relates to whether we want LLMs to produce normative outputs, representing an idealized, stereotype-free world, or descriptive outputs, reflecting observed social patterns.  

Now discuss with your classmates: **Where should the balance lie?**
:::

If you find this question and/or topic interesting, I **highly** recommend:
:::{admonition} PAPER SPOTLIGHT: Bias in LLMs is a feature, not a bug {cite:p}`resnik_large_2025`
:class: fuchsia, dropdown
[Large Language Models Are Biased Because They Are Large Language Models](https://direct.mit.edu/coli/article/51/3/885/128621/Large-Language-Models-Are-Biased-Because-They-Are).

**I especially recommend section 5 for an explanation of descriptive versus normative bias.** The paper also presents a perspective that bias in LLMs is a fundamental feature, not a bug. Resik argues that we would need to consider creating "LLM-like technologies" where the bias is not baked into the model architecture.
:::

:::{admonition} PAPER SPOTLIGHT: Sometimes group discrimination is fair {cite:p}`wang_fairness_2025`
:class: fuchsia, dropdown
[Fairness through Difference Awareness: Measuring Desired Group Discrimination in LLMs](https://aclanthology.org/2025.acl-long.341.pdf)

The presents a problem in current bias literature: Discrimination is often conflated with the idea that everyone should be treated equally, but in some cases we may want to treat groups differently to *actually* make it fair. {cite:t}`wang_fairness_2025` introduces a framework for how to handle this. They won `best paper` at ACL205 !
:::